### Parsing CSV and Excel Files

In [1]:
import pandas as pd

In [2]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

/tmp/ipykernel_808/731168459.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [3]:
# Method - 1: CSVLoader - Each row a document
print(f"CSVLoader - Row based documents")
csv_loader = CSVLoader(
    file_path = "data/structured_files/products.csv",
    encoding = 'utf-8',
    csv_args = {
        'delimiter':',',
        'quotechar':'"',
    }
)
csv_docs = csv_loader.load()
print(f"Loaded {len(csv_docs)} documents (one per row)")
print("\nFirst Document: ")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

CSVLoader - Row based documents
Loaded 5 documents (one per row)

First Document: 
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


### Custom CSV Processing

In [4]:
from typing import List
from langchain_core.documents import Document
# Method-2: Custom csv processing for better control
print("\n Custom Csv Processing")
def process_csv_intelligently(filepath: str) -> List[Document]:
    df = pd.read_csv(filepath)
    documents = []
    # strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        # create structured content
        content = f"""Product Information:
        Name : {row['Product']}
        Category : {row['Category']}
        price : ${row['Price']}
        Stock : {row['Stock']}
        Description : {row['Description']}"""

        # Create document with rich metadata
        doc = Document(
            page_content = content,
            metadata = {
                'source':filepath,
                'row_index': idx,
                'product_name': row['Product'],
                'category':row['Category'],
                'price': row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
    return documents


 Custom Csv Processing


In [5]:
process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name : Laptop\n        Category : Electronics\n        price : $999.99\n        Stock : 50\n        Description : High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name : Mouse\n        Category : Accessories\n        price : $29.99\n        Stock : 200\n        Description : Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'data_type': 'product_info'}, page_content='Product Informati

### Excel Processing

In [7]:
# Method 1: Using pandas for full control
print("Pandas-based Excel processing ")
def process_excel_with_pandas(filepath: str)->List[Document]:
    documents = []
    excel_file = pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet_name)

        # create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {','.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content = sheet_content,
            metadata = {
                'source':filepath,
                'sheet_name':sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_types': 'excel_sheet'
            }
        )
        documents.append(doc)
    return documents 

Pandas-based Excel processing 


In [ ]:
excel_docs = process_excel_with_pandas('data/structured_files/inventory.xlsx')
print(f"Processed {len(excel_docs)} sheets")